# **[PROJECT 1] Day4 — LangGraph 기반 에이전틱 워크플로**

============================================================  
[프로젝트] 2026년 2차 서울시 청년안심주택(공공임대) 청약 도우미  
------------------------------------------------------------  
| 일차 | 역할 | 데이터 |
|------|------|--------|
| day2 | PDF 공고문 RAG | 모집공고문 PDF (Qdrant) |
| day3 | 테이블 Text2SQL | CSV 3종 (SQLite/Supabase) |
| **day4** | **의도 라우팅 에이전트** | RAG + Text2SQL 통합 |

08 `LangGraph essentials`(State · Node · Edge)와  
09 `conditional edges`(의도 분류 후 분기)를 같은 방식으로 적용합니다.

## 과제 진행 단계
1. **에이전트 목적 및 결과물 설계** — Use Case 정의
2. **시스템에 필요한 단계 정의** — 베이스라인 대비 추가 노드
3. **노드와 엣지 추가 및 고도화** — 조건부 라우팅 + 재시도 루프

## 0. 환경 설정

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"✓ .env 로드: {dotenv_path}")
else:
    load_dotenv()

print("\n=== 환경 변수 확인 ===")
print("✓ OpenAI API Key" if os.environ.get("OPENAI_API_KEY") else "✗ OPENAI_API_KEY 없음")
print("✓ Qdrant" if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY") else "✗ Qdrant 미설정")
print("✓ Supabase DB URL" if os.environ.get("SUPABASE_DB_URL") else "- SUPABASE_DB_URL (SQLite fallback 가능)")

In [ ]:
CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "src" / "ai").exists() else CWD.parent
SRC_DIR = REPO_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"src 경로 추가: {SRC_DIR}")

---

# 1. 에이전트 목적 및 결과물 설계

## 해결하려는 문제
66페이지 공고문과 단지·자격 표가 흩어져 있어, 지원자가 일정·자격·임대조건·서류를 한 번에 찾기 어렵습니다.
질문은 **문서 조항(RAG)** 과 **숫자/표 조회(Text2SQL)** 로 나뉘므로 의도를 먼저 분류한 뒤 경로를 나눕니다.

## 결과물
- LangGraph 에이전트 (`src/ai`)
- 이 노트북에서 그래프 컴파일 · 시각화 · 경로별 테스트

In [ ]:
PROJECT_TOPIC = "2026년 2차 서울시 청년안심주택(공공임대) 청약 도우미"

USE_CASES = [
    {"intent": "general", "name": "서비스 안내", "question": "안녕하세요, 뭘 물어볼 수 있나요?"},
    {"intent": "vector", "name": "청약 일정", "question": "청약 접수는 언제부터 언제까지인가요?"},
    {"intent": "vector", "name": "제출서류", "question": "청년 2순위로 신청하려면 어떤 서류가 필요한가요?"},
    {"intent": "vector", "name": "유의사항", "question": "반려동물 키울 수 있나요?"},
    {"intent": "database", "name": "단지 임대조건", "question": "에이트플레이스 39A 타입 신혼Ⅰ 임대보증금과 월세는 얼마인가요?"},
    {"intent": "database", "name": "자격요건 표", "question": "청년 1순위 자격요건에 해당하는 경우는 어떤 경우들이 있나요?"},
    {"intent": "database", "name": "문의처", "question": "청년안심주택 민간임대 계약 관련해서 어디로 전화해야 하나요?"},
]

print(f"프로젝트: {PROJECT_TOPIC}\n")
for i, uc in enumerate(USE_CASES, 1):
    print(f"{i}. [{uc['intent']}] {uc['name']}")
    print(f"   Q. {uc['question']}")

---

# 2. 시스템에 필요한 단계 정의

09의 의도 분류 챗봇 베이스라인(`analyze_intent` → 응답 노드 → END)에서 아래를 추가합니다.

| 단계 | 노드 | 왜 필요한가 |
|------|------|-------------|
| 의도 분류 | `classify_intent` | 인사 / DB 수치 / 공고문 조항을 갈라야 함 |
| 일반 응답 | `general_answer` | RAG·SQL 없이 바로 답 |
| 문서 검색 | `vector_search` | day2 Qdrant + 카테고리 필터 |
| 쿼리 재작성 | `rewrite_query` | 검색 실패 시 한 번 더 시도 |
| 표 조회 | `database_query` | day3 Text2SQL, 오류 시 재시도 |
| 최종 답변 | `generate_answer` | 검색/SQL 근거를 자연어로 정리 |

```
__start__
   ↓
classify_intent
   ├── general_answer → __end__
   ├── database_query ↺ → generate_answer → __end__
   └── vector_search ↔ rewrite_query
              ↓
         generate_answer → __end__
```

## 2-1. Text2SQL용 SQLite 확인

day3 CSV(`apartment`, `preferences`, `service_center`)를 로컬 DB로 사용합니다.

In [ ]:
import sqlite3
import pandas as pd
from langchain_community.utilities import SQLDatabase

DATA_CANDIDATES = [
    REPO_ROOT / "project" / "day3",
    REPO_ROOT.parent / "smu-ai-service-bootcamp" / "rag-system" / "project" / "day3",
    REPO_ROOT.parent / "smu-ai-service-bootcamp" / "rag-system" / "SMU-AI-BOOTCAMP" / "day3",
]
DATA_DIR = next((p for p in DATA_CANDIDATES if (p / "apartment.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("apartment.csv를 찾지 못했습니다. project/day3를 확인하세요.")

DB_PATH = DATA_DIR / "database" / "youth_housing.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

APARTMENT_RENAME = {
    "구분": "category", "단지명": "complex_name", "주소": "address",
    "공급유형(㎡)": "supply_area_type", "평면유형(㎡)": "floor_plan_area",
    "신청자격": "eligibility", "공급호수(실)": "units", "임대료구분": "rent_grade",
    "임대보증금_계(천원)": "deposit_total_k", "계약금_20%(천원)": "contract_deposit_k",
    "잔금_80%(천원)": "balance_k", "월임대료(원)": "monthly_rent",
}
PREFERENCES_RENAME = {
    "신청자격": "eligibility", "구분유형": "category_type", "코드": "code",
    "세부항목": "item", "기준값": "criteria_value", "비고": "note",
}
SERVICE_CENTER_RENAME = {
    "구분": "category", "기관/부서명": "organization", "담당업무": "service",
    "전화번호": "phone", "비고": "note",
}

conn = sqlite3.connect(DB_PATH)
for table, rename in [
    ("apartment", APARTMENT_RENAME),
    ("preferences", PREFERENCES_RENAME),
    ("service_center", SERVICE_CENTER_RENAME),
]:
    df = pd.read_csv(DATA_DIR / f"{table}.csv").rename(columns=rename)
    df.to_sql(table, conn, if_exists="replace", index=False)
    print(f"✓ {table}: {len(df)}행")
conn.close()

os.environ["SQLITE_DB_PATH"] = str(DB_PATH)
db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH.as_posix()}")
print(f"\n✓ SQLite: {DB_PATH}")
print(f"테이블: {db.get_usable_table_names()}")

---

# 3. 노드와 엣지 추가 및 고도화

## 3-1. State (08 essentials)

`MessagesState`로 대화 히스토리를 누적하고, 검색·SQL 중간값을 필드로 둡니다.

In [ ]:
from ai.state import AgentState, InputState

print("AgentState 필드:")
for name, typ in AgentState.__annotations__.items():
    print(f"  - {name}: {typ}")

## 3-2. 노드와 라우팅 함수 (09 conditional edges)

구현은 `src/ai/nodes.py`에 있습니다. 노트북에서는 그래프를 08·09와 같이 조립합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage

from ai.nodes import (
    classify_intent,
    general_answer,
    vector_search,
    database_query,
    rewrite_query,
    generate_answer,
    route_by_intent,
    check_vector_results,
    check_db_results,
)

graph_builder = StateGraph(AgentState, input=InputState)

graph_builder.add_node("classify_intent", classify_intent)
graph_builder.add_node("general_answer", general_answer)
graph_builder.add_node("vector_search", vector_search)
graph_builder.add_node("database_query", database_query)
graph_builder.add_node("rewrite_query", rewrite_query)
graph_builder.add_node("generate_answer", generate_answer)

graph_builder.add_edge(START, "classify_intent")

graph_builder.add_conditional_edges(
    "classify_intent",
    route_by_intent,
    {
        "general_answer": "general_answer",
        "database_query": "database_query",
        "vector_search": "vector_search",
    },
)

graph_builder.add_edge("general_answer", END)

graph_builder.add_conditional_edges(
    "vector_search",
    check_vector_results,
    {
        "generate_answer": "generate_answer",
        "rewrite_query": "rewrite_query",
    },
)
graph_builder.add_edge("rewrite_query", "vector_search")

graph_builder.add_conditional_edges(
    "database_query",
    check_db_results,
    {
        "generate_answer": "generate_answer",
        "database_query": "database_query",
    },
)

graph_builder.add_edge("generate_answer", END)

graph = graph_builder.compile()
print("✓ 그래프 컴파일 완료")

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("시각화 생략:", e)
    print(graph.get_graph().draw_mermaid())

## 3-3. 경로별 테스트

`invoke`로 한 경로씩 확인하고, 의도와 최종 답변을 출력합니다.

In [ ]:
def run_agent(question: str):
    result = graph.invoke({"messages": [HumanMessage(content=question)]})
    answer = result["messages"][-1].content
    print(f"Q. {question}")
    print(f"intent: {result.get('intent')}")
    if result.get("sql_query"):
        print(f"SQL:\n{result['sql_query']}")
    if result.get("rewritten_query"):
        print(f"rewritten_query: {result['rewritten_query']}")
    n_docs = len(result.get("vector_results") or [])
    if n_docs:
        print(f"vector_results: {n_docs}개")
    print(f"A. {answer}\n")
    print("-" * 60)
    return result

In [ ]:
# 경로 1) general_answer → END
run_agent("안녕하세요, 뭘 물어볼 수 있나요?")

In [ ]:
# 경로 2) vector_search (↔ rewrite_query) → generate_answer → END
run_agent("청약 접수는 언제부터 언제까지인가요?")

In [ ]:
# 경로 3) database_query (재시도 가능) → generate_answer → END
run_agent("에이트플레이스 39A 타입 신혼Ⅰ 임대보증금과 월세는 얼마인가요?")

In [ ]:
run_agent("청년안심주택 민간임대 계약 관련해서 어디로 전화해야 하나요?")

## 3-4. 패키지 그래프와 동일성

LangGraph Studio는 `src/ai/graph.py:graph`를 사용합니다. 위 셀과 같은 워크플로입니다.

In [ ]:
from ai.graph import create_graph

studio_graph = create_graph()
print("노드:", list(studio_graph.get_graph().nodes))
print("\n프로젝트 구조")
print("src/ai/")
print("  state.py      # AgentState")
print("  nodes.py      # classify / search / sql / answer")
print("  retriever.py  # Qdrant (cheongnyeon_anshim_housing)")
print("  text2sql.py   # apartment · preferences · service_center")
print("  graph.py      # 조건부 엣지 워크플로")